# Trade Tape Analysis & Markout
Analyze fills, adverse selection, and markout PnL from the reward farmer.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

DB_PATH = 'trades.db'
db = sqlite3.connect(DB_PATH)
df = pd.read_sql('SELECT * FROM trades ORDER BY ts', db)
db.close()

df['dt'] = pd.to_datetime(df['dt'])
print(f'{len(df)} trades from {df.dt.min()} to {df.dt.max()}')
df.head()

In [ ]:
# ── Separate our fills vs market tape ──
fills = df[df['side'].str.contains('FILL')].copy()
tape = df[~df['side'].str.contains('FILL')].copy()

print(f'Our fills: {len(fills)}')
print(f'Market tape: {len(tape)}')
print()
print('Fills by market:')
fills.groupby('market').agg(
    count=('size', 'count'),
    total_shares=('size', 'sum'),
    avg_price=('price', lambda x: np.average(x, weights=fills.loc[x.index, 'size'])),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    total_cost=('price', lambda x: (x * fills.loc[x.index, 'size']).sum()),
).round(3)

In [ ]:
# ── Markout analysis ──
# For each fill, look at what the market price was N seconds later
# This tells us if we got adversely selected

MARKOUT_WINDOWS = [5, 10, 30, 60, 120, 300]  # seconds

def compute_markouts(fills_df, tape_df):
    """For each fill, find the tape mid price at fill_time + N seconds."""
    results = []
    
    for _, fill in fills_df.iterrows():
        market = fill['market']
        fill_ts = fill['ts']
        fill_px = fill['price']
        fill_sz = fill['size']
        fill_side = fill['side']
        
        # Get tape for this market (use price range to match YES vs NO)
        # Match by price proximity — fills < 0.5 match tape < 0.5
        if fill_px < 0.5:
            mkt_tape = tape_df[(tape_df['price'] < 0.6)].copy()
        else:
            mkt_tape = tape_df[(tape_df['price'] > 0.4)].copy()
        
        row = {
            'market': market,
            'fill_time': fill['dt'],
            'fill_price': fill_px,
            'fill_size': fill_sz,
            'fill_side': fill_side,
            'fill_cost': fill_px * fill_sz,
        }
        
        for window in MARKOUT_WINDOWS:
            future_trades = mkt_tape[
                (mkt_tape['ts'] > fill_ts) & 
                (mkt_tape['ts'] <= fill_ts + window)
            ]
            if len(future_trades) > 0:
                # Volume-weighted mid of future trades
                future_mid = np.average(future_trades['price'], weights=future_trades['size'])
                markout = future_mid - fill_px  # positive = price went up after we bought
                markout_pnl = markout * fill_sz
            else:
                future_mid = np.nan
                markout = np.nan
                markout_pnl = np.nan
            
            row[f'mid_{window}s'] = future_mid
            row[f'markout_{window}s'] = markout
            row[f'pnl_{window}s'] = markout_pnl
        
        results.append(row)
    
    return pd.DataFrame(results)

markouts = compute_markouts(fills, tape)
print(f'Computed markouts for {len(markouts)} fills')
markouts.head()

In [ ]:
# ── Markout summary by market ──
# Average markout at each window — negative = adverse selection

markout_cols = [c for c in markouts.columns if c.startswith('markout_')]
pnl_cols = [c for c in markouts.columns if c.startswith('pnl_')]

print('=== Average Markout (cents) by Market ===')
print('Negative = price moved against us after fill (adverse selection)')
print()
summary = markouts.groupby('market')[markout_cols].mean() * 100  # convert to cents
print(summary.round(2).to_string())

print()
print('=== Total Markout PnL ($) by Market ===')
pnl_summary = markouts.groupby('market')[pnl_cols].sum()
print(pnl_summary.round(2).to_string())

print()
print('=== Overall ===')
print('Avg markout (cents):', (markouts[markout_cols].mean() * 100).round(2).to_string())
print('Total PnL ($):', markouts[pnl_cols].sum().round(2).to_string())

In [ ]:
# ── Chart 1: Markout curves by market ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

windows = MARKOUT_WINDOWS

# Left: average markout in cents
ax = axes[0]
for market, group in markouts.groupby('market'):
    avgs = [group[f'markout_{w}s'].mean() * 100 for w in windows]
    label = market[:25]
    ax.plot(windows, avgs, 'o-', label=label, markersize=4)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Markout Window (seconds)')
ax.set_ylabel('Average Markout (cents)')
ax.set_title('Markout Curves by Market')
ax.legend(fontsize=7, loc='best')
ax.grid(True, alpha=0.3)

# Right: cumulative PnL
ax = axes[1]
for market, group in markouts.groupby('market'):
    totals = [group[f'pnl_{w}s'].sum() for w in windows]
    label = market[:25]
    ax.plot(windows, totals, 'o-', label=label, markersize=4)
ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Markout Window (seconds)')
ax.set_ylabel('Total Markout PnL ($)')
ax.set_title('Cumulative Markout PnL by Market')
ax.legend(fontsize=7, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 2: Price + fills timeline per market ──
# Pick the market with most fills

top_markets = fills.groupby('market')['size'].sum().sort_values(ascending=False).head(4).index

fig, axes = plt.subplots(len(top_markets), 1, figsize=(16, 4*len(top_markets)), sharex=False)
if len(top_markets) == 1:
    axes = [axes]

for ax, market in zip(axes, top_markets):
    mkt_fills = fills[fills['market'] == market]
    
    # Get tape trades in similar price range
    avg_fill_px = mkt_fills['price'].mean()
    if avg_fill_px < 0.5:
        mkt_tape = tape[tape['price'] < 0.6]
    else:
        mkt_tape = tape[tape['price'] > 0.4]
    
    # Market tape
    buys = mkt_tape[mkt_tape['side'] == 'BUY']
    sells = mkt_tape[mkt_tape['side'] == 'SELL']
    ax.scatter(buys['dt'], buys['price'], c='lightgreen', s=2, alpha=0.3, label='Tape BUY')
    ax.scatter(sells['dt'], sells['price'], c='lightcoral', s=2, alpha=0.3, label='Tape SELL')
    
    # Our fills
    sizes = mkt_fills['size'].clip(upper=2000)
    ax.scatter(mkt_fills['dt'], mkt_fills['price'], 
              c='blue', s=sizes/20 + 10, alpha=0.8, 
              edgecolors='navy', label='Our fills', zorder=5)
    
    # Running avg
    cum_cost = (mkt_fills['price'] * mkt_fills['size']).cumsum()
    cum_sz = mkt_fills['size'].cumsum()
    ax.plot(mkt_fills['dt'], cum_cost / cum_sz, 'b--', linewidth=1.5, label='Avg price')
    
    ax.set_title(f'{market[:40]} — {len(mkt_fills)} fills, {mkt_fills["size"].sum():.0f} shares')
    ax.set_ylabel('Price')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 3: Fill size distribution ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(fills['size'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Fill Size (shares)')
ax.set_ylabel('Count')
ax.set_title('Fill Size Distribution')
ax.axvline(fills['size'].median(), color='red', linestyle='--', label=f'Median: {fills["size"].median():.0f}')
ax.axvline(fills['size'].mean(), color='orange', linestyle='--', label=f'Mean: {fills["size"].mean():.0f}')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(fills['price'], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Fill Price')
ax.set_ylabel('Count')
ax.set_title('Fill Price Distribution')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 4: Markout by fill size (are bigger fills more adversely selected?) ──
fig, ax = plt.subplots(figsize=(10, 6))

size_bins = pd.qcut(markouts['fill_size'], q=5, duplicates='drop')
for window in [10, 30, 60, 300]:
    col = f'markout_{window}s'
    grouped = markouts.groupby(size_bins)[col].mean() * 100
    ax.plot(range(len(grouped)), grouped.values, 'o-', label=f'{window}s markout')
    ax.set_xticks(range(len(grouped)))
    ax.set_xticklabels([str(x)[:15] for x in grouped.index], rotation=20, fontsize=8)

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Fill Size Bucket')
ax.set_ylabel('Average Markout (cents)')
ax.set_title('Markout by Fill Size — Are bigger fills more toxic?')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Chart 5: Cumulative PnL over time ──
fig, ax = plt.subplots(figsize=(14, 6))

for window in [10, 60, 300]:
    col = f'pnl_{window}s'
    valid = markouts.dropna(subset=[col]).sort_values('fill_time')
    cum_pnl = valid[col].cumsum()
    ax.plot(valid['fill_time'], cum_pnl, label=f'{window}s markout PnL')

ax.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax.set_xlabel('Time')
ax.set_ylabel('Cumulative Markout PnL ($)')
ax.set_title('Cumulative Markout PnL Over Time')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Pair analysis: YES + NO cost vs $1 settle ──
yes_fills = fills[fills['price'] < 0.5].copy()
no_fills = fills[fills['price'] >= 0.5].copy()

yes_cost = (yes_fills['price'] * yes_fills['size']).sum()
yes_shares = yes_fills['size'].sum()
yes_avg = yes_cost / yes_shares if yes_shares > 0 else 0

no_cost = (no_fills['price'] * no_fills['size']).sum()
no_shares = no_fills['size'].sum()
no_avg = no_cost / no_shares if no_shares > 0 else 0

mergeable = min(yes_shares, no_shares)
pair_cost = yes_avg + no_avg
spread_loss = (pair_cost - 1.0) * mergeable

print('=== Pair Cost Analysis ===')
print(f'YES: {yes_shares:.0f} shares @ avg {yes_avg:.4f} = ${yes_cost:.2f}')
print(f'NO:  {no_shares:.0f} shares @ avg {no_avg:.4f} = ${no_cost:.2f}')
print(f'\nPair cost: {yes_avg:.4f} + {no_avg:.4f} = {pair_cost:.4f}')
print(f'Mergeable: {mergeable:.0f} pairs')
print(f'Spread loss: ${spread_loss:.2f} ({(pair_cost-1)*100:.1f}¢ per pair)')
print(f'\nBreakeven daily reward needed: ${abs(spread_loss):.2f}/day')